# Gradient Boosting: Step-by-Step Solutions

This notebook contains the questions and complete worked solutions for the Gradient Boosting exercise (Q.1 – Q.4).

## Q.1 — Initial Prediction & Residuals

**Dataset**  
\( x = [1, 2, 3, 4, 5] \)  
\( y = [2.5, 3.5, 3.0, 5.5, 6.0] \) (regression target)

### 01 & 02. Initial prediction
\( F_0(x) = \text{mean}(y) \) for all points.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1., 2., 3., 4., 5.])
y = np.array([2.5, 3.5, 3.0, 5.5, 6.0])

F0 = np.mean(y)
print(f"F0 (initial prediction) = {F0:.4f}")


### 03. Compute residuals
\( r_{1i} = y_i - F_0(x_i) \)


In [ ]:
r1 = y - F0
print("Residual vector r1 =", np.round(r1, 4))


### 04. Fit a decision stump on residuals
Split at \( x \le 2 \).  
Predict mean residual of the left group for left, mean residual of the right group for right.


In [ ]:
left_mask = x <= 2
right_mask = x > 2

mean_left = np.mean(r1[left_mask])
mean_right = np.mean(r1[right_mask])

print(f"Mean residual (x ≤ 2) = {mean_left:.4f}")
print(f"Mean residual (x > 2) = {mean_right:.4f}")

h1 = np.where(x <= 2, mean_left, mean_right)
print("h1(x) predictions =", np.round(h1, 4))


### 05. Update model
\( F_1(x) = F_0(x) + \eta \, h_1(x) \) with learning rate \( \eta = 0.5 \).


In [ ]:
eta = 0.5
F1 = F0 + eta * h1
print("F1 for each x =", np.round(F1, 4))


### 06. Compute new residuals and compare magnitude
\( r_{2i} = y_i - F_1(x_i) \)


In [ ]:
r2 = y - F1
print("New residual vector r2 =", np.round(r2, 4))
print()
print(f"Mean |r1| = {np.mean(np.abs(r1)):.4f}")
print(f"Mean |r2| = {np.mean(np.abs(r2)):.4f}")
print("→ Residuals have become smaller on average (as expected).")


## Q.2 — Second Iteration & Classification Variant

### 01. Fit second stump on r₂
Split at \( x \le 3 \). Compute mean(r₂) for each group.


In [ ]:
left2 = r2[x <= 3]
right2 = r2[x > 3]

mean_left2 = np.mean(left2)
mean_right2 = np.mean(right2)

print(f"Mean residual (x ≤ 3) = {mean_left2:.4f}")
print(f"Mean residual (x > 3) = {mean_right2:.4f}")

h2 = np.where(x <= 3, mean_left2, mean_right2)
print("h2(x) predictions =", np.round(h2, 4))


### 02. Update model
\( F_2(x) = F_1(x) + \eta \, h_2(x) \) with \( \eta = 0.5 \).


In [ ]:
F2 = F1 + eta * h2
print("F2 for each x =", np.round(F2, 4))


### 03. Evaluate MSE for F₀, F₁ and F₂


In [ ]:
mse0 = np.mean((y - F0)**2)
mse1 = np.mean((y - F1)**2)
mse2 = np.mean((y - F2)**2)

print(f"MSE(F0) = {mse0:.4f}")
print(f"MSE(F1) = {mse1:.4f}")
print(f"MSE(F2) = {mse2:.4f}")
print("→ MSE decreases with each boosting round.")


### 04–05. Classification variant
Convert target to binary: \( y = [0, 0, 0, 1, 1] \).  
Use log-loss.  
Initial prediction \( p_0 = \text{mean}(y) \).  
Pseudo-residuals under log-loss: \( r_{1i} = y_i - p_0 \).


In [ ]:
y_bin = np.array([0., 0., 0., 1., 1.])
p0 = np.mean(y_bin)
print(f"p0 (initial probability) = {p0:.4f}")

r1_class = y_bin - p0
print("Pseudo-residuals r1 =", np.round(r1_class, 4))


### 06. Reflection
**What does the pseudo-residual represent in terms of model error?**

Under the logistic (log-loss) objective, the negative gradient of the loss with respect to the current prediction is exactly \( y_i - p_i \).  
Thus the pseudo-residual is the direction and magnitude of the error that the next weak learner should correct: positive when the true label is 1 and the current probability is too low, negative when the true label is 0 and the current probability is too high.


## Q.3 — Visualize It

We produce the line plot of the original targets together with the successive model predictions, residual bar charts, and a note on the learning-rate effect.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [2, 1]})

# --- Main prediction plot ---
ax = axes[0]
ax.scatter(x, y, s=80, c='black', zorder=5, label='True y')
ax.plot(x, np.full_like(x, F0), 'b--', linewidth=2, label=r'$F_0$ (mean)')
ax.plot(x, F1, 'g-o', linewidth=2, markersize=6, label=r'$F_1$')
ax.plot(x, F2, 'r-s', linewidth=2, markersize=6, label=r'$F_2$')

ax.set_xlabel('x')
ax.set_ylabel('y / prediction')
ax.set_title('Gradient Boosting Predictions vs True Values')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xticks(x)

# Learning-rate note
ax.annotate(
    r'With $\eta=1.0$ the steps would be twice as large → faster fit but higher risk of overshoot.\n'
    r'With $\eta=0.1$ the steps would be 5× smaller → slower, smoother, more regularised convergence.',
    xy=(0.02, 0.02), xycoords='axes fraction',
    fontsize=9, style='italic',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8)
)

# --- Residual bar chart ---
ax2 = axes[1]
width = 0.35
ax2.bar(x - width/2, r1, width, label=r'$r_1$', color='steelblue', alpha=0.8)
ax2.bar(x + width/2, r2, width, label=r'$r_2$', color='coral', alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('x')
ax2.set_ylabel('Residual')
ax2.set_title('Residuals shrink after each boosting step')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xticks(x)

plt.tight_layout()
plt.show()


### Bonus — 3-panel figure


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: F0 & data
axes[0].scatter(x, y, s=80, c='black', zorder=5)
axes[0].axhline(F0, color='blue', linestyle='--', linewidth=2, label=r'$F_0$')
axes[0].set_title(r'Panel 1: Data & $F_0$')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(x)

# Panel 2: stump h1
axes[1].step(x, h1, where='mid', color='green', linewidth=2, label=r'$h_1$')
axes[1].scatter(x, r1, s=60, c='steelblue', zorder=5, label=r'$r_1$')
axes[1].axhline(0, color='gray', linewidth=0.8)
axes[1].set_title(r'Panel 2: Stump $h_1$ fitted on $r_1$')
axes[1].set_xlabel('x')
axes[1].set_ylabel('residual / prediction')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(x)

# Panel 3: F1 corrected curve
axes[2].scatter(x, y, s=80, c='black', zorder=5, label='True y')
axes[2].plot(x, F1, 'g-o', linewidth=2, markersize=6, label=r'$F_1$')
axes[2].set_title(r'Panel 3: Corrected model $F_1$')
axes[2].set_xlabel('x')
axes[2].set_ylabel('prediction')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_xticks(x)

plt.tight_layout()
plt.show()


## Q.4 — Explain It Cold

### 01. Core idea of Gradient Boosting
Each new tree is fitted to the **negative gradient** (i.e., the residual / pseudo-residual) of the loss with respect to the current ensemble prediction.  
By repeatedly adding a weak learner that points in the direction of steepest descent of the loss, the ensemble gradually reduces the training loss.  
In other words, every new tree is trying to correct the mistakes of the current model.

### 02. Role of the learning rate \( \eta \)
\( \eta \) scales the contribution of each new tree:
- When \( \eta = 1.0 \) the full correction suggested by the tree is added → faster reduction of training loss but higher risk of overshooting and overfitting.
- When \( \eta \to 0 \) each step becomes tiny → the model converges very slowly and needs many more trees, but the path is much smoother and the final model is more regularised (less prone to overfitting).

### 03. Regression (MSE) vs Classification (log-loss)
**What stays the same**
- The overall boosting framework (additive model, sequential fitting of weak learners).
- The use of a learning rate and the additive update \( F_m = F_{m-1} + \eta\, h_m \).

**What changes**
- The **loss function** and therefore the **quantity the trees are fitted to**:
  - MSE → ordinary residuals \( y - F \).
  - Log-loss → pseudo-residuals \( y - p \) (where \( p = \sigma(F) \) is the predicted probability).
- The final prediction interpretation (continuous value vs probability / log-odds).

### 04. Risk of too many boosting rounds & regularisation
Too many rounds can drive training loss to (near) zero and cause **severe overfitting**.  

Regularisation mechanisms that mitigate this:
- **Small learning rate \( \eta \)** – forces the model to take many tiny steps, making it harder to fit noise.
- **Limited tree depth** (and other tree constraints such as min samples per leaf) – keeps each individual learner weak, so the ensemble cannot memorise the training set too quickly.
- Early stopping based on a validation set is also commonly used.


---
### Summary of numerical results

| Quantity | Value |
|----------|-------|
| \( F_0 \) | 4.1 |
| \( r_1 \) | [-1.6, -0.6, -1.1, 1.4, 1.9] |
| \( h_1 \) (split ≤ 2) | [-1.1, -1.1, 0.7333, 0.7333, 0.7333] |
| \( F_1 \) | [3.55, 3.55, 4.4667, 4.4667, 4.4667] |
| \( r_2 \) | [-1.05, -0.05, -1.4667, 1.0333, 1.5333] |
| \( h_2 \) (split ≤ 3) | [-0.8556, -0.8556, -0.8556, 1.2833, 1.2833] |
| \( F_2 \) | [3.1222, 3.1222, 4.0389, 5.1083, 5.1083] |
| MSE(F₀) | 1.9400 |
| MSE(F₁) | 1.3350 |
| MSE(F₂) | 0.5115 |
| Classification \( p_0 \) | 0.4 |
| Pseudo-residuals | [-0.4, -0.4, -0.4, 0.6, 0.6] |
